This test is on
1. i.i.d. rayleigh fading channel
2. 4*4 MIMO channel
3. 1000 symbols per channel
4. total of 1000 channels


In [2]:
"""
DONT RUN THIS CODE BLOCK WHEN USING LOCAL COMPUTER
THIS IS ONLY FOR GOOGLE COLAB
"""
!git clone https://github.com/billlllllllly/EE_MIMO_PROJECT.git
# load the whole repo
%cd EE_MIMO_PROJECT
# change the dir
!ls
# just for checking that the current folder is correct

Cloning into 'EE_MIMO_PROJECT'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 154 (delta 43), reused 105 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 811.89 KiB | 11.60 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/content/EE_MIMO_PROJECT/EE_MIMO_PROJECT
NYCU_scene  src


In [3]:
import sys
from pathlib import Path

# cwd = Path.cwd()
# repo_root = next(path for path in (cwd, *cwd.parents) if (path / "src/models/MAP.py").is_file())
# sys.path.insert(0, str(repo_root))

# import models
from src.models.MAP import MAP
from src.models.MMSE import MMSE
from src.models.SIC import SIC
from src.models.partialMAP import partialMAP

models = {
    "MAP": MAP(),
    "MMSE": MMSE(),
    "QR-SIC": SIC(),
    "Partial MAP": partialMAP()
}

In [4]:
# mimo settings
import torch
from src.channel.rayleighfading import rayleighfading
BLOCKS = 100000
SYMBOLS_PER_CHANNEL = 1
# total of (SYMBOLS_PER_CHANNEL * BLOCKS) symbols are transmitted
NT = 4
NR = 4
SNR_DB = list(range(0, 22, 2))

def generateChannel(NR, NT):
    return rayleighfading(NR=NR, NT=NT)

qpsk = torch.tensor([-1-1j, -1+1j, 1+1j, 1-1j], dtype=torch.complex128) / 2**0.5
gray = torch.tensor([[0, 0], [1, 0], [1, 1], [0, 1]])

In [5]:
# channel simulation
for detector in models.values():
    detector.updatePossibleTXSymbols(qpsk, gray)

errors = torch.zeros((len(SNR_DB), len(models)), dtype=torch.long)
bits_per_snr = gray.shape[1] * NT * SYMBOLS_PER_CHANNEL * BLOCKS

for i in range(BLOCKS):
    T = 1000
    if (i % T == 0):
        print(f"currently running block: {i} to {i+T}")
    CSI = generateChannel(NT=NT, NR=NR).to(qpsk.dtype)
    tx_idx = torch.randint(len(qpsk), (NT, SYMBOLS_PER_CHANNEL))
    tx_bits = gray[tx_idx]
    y_clean = CSI @ qpsk[tx_idx]
    noise_base = torch.randn_like(y_clean)

    for detector in models.values():
        detector.updateCSI(CSI)

    for i, snr in enumerate(SNR_DB):
        n0 = 10.0 ** (-snr / 10)
        received = (y_clean + n0**0.5 * noise_base).T
        for j, detector in enumerate(models.values()):
            detector.updateNoiseVar(n0)
            detected_bits = detector.detectSequence(received)
            errors[i, j] += (detected_bits != tx_bits).count_nonzero()

ber = errors.double() / bits_per_snr
print(f"{bits_per_snr:,} bits per SNR")
print("SNR (dB)   " + "  ".join(f"{name:>12}" for name in models))
for snr, row in zip(SNR_DB, ber):
    print(f"{snr:8g}   " + "  ".join(f"{value:12.4g}" for value in row))

currently running block: 0 to 10
currently running block: 10 to 20
currently running block: 20 to 30
currently running block: 30 to 40
currently running block: 40 to 50
currently running block: 50 to 60
currently running block: 60 to 70
currently running block: 70 to 80
currently running block: 80 to 90
currently running block: 90 to 100
currently running block: 100 to 110
currently running block: 110 to 120
currently running block: 120 to 130
currently running block: 130 to 140
currently running block: 140 to 150
currently running block: 150 to 160
currently running block: 160 to 170
currently running block: 170 to 180
currently running block: 180 to 190
currently running block: 190 to 200
currently running block: 200 to 210
currently running block: 210 to 220
currently running block: 220 to 230
currently running block: 230 to 240
currently running block: 240 to 250
currently running block: 250 to 260
currently running block: 260 to 270
currently running block: 270 to 280
currently ru

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from google.colab import files

snr_db = np.asarray(SNR_DB)
fig, ax = plt.subplots(figsize=(8, 5))

for j, name in enumerate(models):
    points = ber[:, j]
    line, = ax.semilogy(
        snr_db, np.where(points > 0, points, np.nan), "o-", label=name
    )

    zero = points == 0
    if zero.any():
        ax.semilogy(
            snr_db[zero],
            np.full(zero.sum(), 1 / bits_per_snr),
            "v",
            color=line.get_color(),
        )

TITLE = f"{NT}by{NR}, Gray QPSK, Rayleigh, {SYMBOLS_PER_CHANNEL} SperH, {BLOCKS} H"
ax.set(xlabel="SNR (dB)", ylabel="BER",
       title=TITLE)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
print("Downward triangles: zero observed errors, shown at 1 / bits per SNR.")

plt.savefig(f'{TITLE}.png', dpi=300, bbox_inches='tight')
files.download(f'{TITLE}.png')

plt.show()